# Module N - Combined Feature Engine

Combines Module L RGB features, Module M AI-estimated spectral features, and optional numerical temporal changes for future ML.

## Objective and safety

This is not a model or diagnostic system. Temporal values describe feature changes only; they do not establish disease progression.

In [ ]:
from pathlib import Path
import sys
from datetime import datetime, timedelta
import numpy as np

root = Path.cwd().resolve()
if not (root / 'src').exists(): root = root.parent
sys.path.insert(0, str(root / 'src'))
from spectraderm.features.rgb_features import RGBFeatureResult
from spectraderm.features.spectral_features import SpectralFeatureResult
from spectraderm.features.combined_features import combine_features

## Create small Module L and Module M-style results

These are clearly labelled synthetic local examples; no MST++ inference is run.

In [ ]:
def rgb_result(value):
    return RGBFeatureResult({'color_mean_r': value, 'gradient_mean': 0.2}, {}, 4, 1.0, (2,2,3), [])
def spectral_result(value):
    return SpectralFeatureResult({'ratio_540_650': value}, {}, np.arange(31, dtype=float) + value, np.arange(400,701,10), 4, 1.0, (2,2,31), [])
baseline = combine_features(rgb_result(0.4), spectral_result(1.0), metadata={'observation_id':'baseline','subject_key':'anonymous'})
follow_up = combine_features(rgb_result(0.5), spectral_result(1.2), baseline, datetime(2024,1,3), datetime(2024,1,1), metadata={'observation_id':'follow-up','subject_key':'anonymous'})

## Current features, temporal changes, and ML vector

In [ ]:
print('History:', follow_up.metadata.history_available, follow_up.metadata.temporal_status)
print('Current RGB:', {k:v for k,v in follow_up.current_features.items() if k.startswith('rgb.')})
print('Current spectral:', {k:v for k,v in follow_up.current_features.items() if k.startswith('spectral.') and 'signature' not in k})
print('Deltas:', {k:v for k,v in follow_up.temporal_features.items() if k.endswith('.delta')})
print('Relative changes:', {k:v for k,v in follow_up.temporal_features.items() if k.endswith('.relative_change')})
print('Vector length:', len(follow_up.ml_vector))
print(list(zip(follow_up.feature_names[:12], follow_up.ml_vector[:12])))

## Baseline/no-history handling and limitations

`baseline` has raw temporal NaN values, availability flags of zero, and a finite ML vector. Availability distinguishes no history from a true zero change. Module M proxies remain investigational; AI-estimated spectra are not direct hyperspectral measurements.